# Notebook 02: Diagnóstico y Reconstrucción de Esquema
## Proyecto II Parcial — Modelado Avanzado de Base de Datos - 30759
## Integrantes:
- Naomi Obando
- Mauri Tandazo

**Fase 2-3:** Diagnóstico de inconsistencias estructurales, homologación al esquema canónico y cuarentena de archivos no recuperables.

---
### ¿Qué se resuelve aquí?
| Problema detectado | Solución aplicada |
|---|---|
| Columnas con nombres distintos por servicio | Matriz de homologación (column mapping) |
| Columnas ausentes | Se completan con NULL controlado |
| Tipos de datos incompatibles | Conversión segura con cast() |
| FHVHV sin total_amount | Calculado como suma de componentes |
| Archivos no recuperables | Enviados a cuarentena con evidencia |

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))

from src.utils          import generate_process_id, load_config, setup_logger, create_spark_session
from src.extract        import run_extraction_phase
from src.schema_recovery import (
    run_schema_recovery_phase, diagnose_schema,
    apply_canonical_mapping, SERVICE_COLUMN_MAPS, CANONICAL_SCHEMA_FIELDS
)

PROCESS_ID  = generate_process_id()
CONFIG_PATH = '../config/etl_config.yaml'
config      = load_config(CONFIG_PATH)
config['paths'] = {k: f'../{v}' for k, v in config['paths'].items()}
config['paths']['metadata_dir'] = '../metadata'

logger = setup_logger('nb02_schema', '../logs', PROCESS_ID)
spark  = create_spark_session(config)
print(f'process_id: {PROCESS_ID} | Spark {spark.version}')

process_id: ETL-20260623-110509-331BC2BB | Spark 3.5.0


In [2]:
# ── 1. Mostrar matrices de homologación ───────────────────────────
import pandas as pd

print('=== MATRIZ DE HOMOLOGACIÓN AL ESQUEMA CANÓNICO ===')
matrix_rows = []

# Construir la matriz de forma inversa (canónico → por servicio)
canonical_to_service = {}
for service, mapping in SERVICE_COLUMN_MAPS.items():
    for orig, canon in mapping.items():
        if canon not in canonical_to_service:
            canonical_to_service[canon] = {}
        canonical_to_service[canon][service] = orig

matrix_rows = []
for canon_col, services in sorted(canonical_to_service.items()):
    matrix_rows.append({
        'Campo Canónico': canon_col,
        'Yellow Taxi':   services.get('yellow', '— (no existe)'),
        'Green Taxi':    services.get('green',  '— (no existe)'),
        'FHVHV':         services.get('fhvhv',  '— (no existe)'),
    })

matrix_df = pd.DataFrame(matrix_rows)
display(matrix_df.style.set_caption('Matriz de Homologación de Columnas'))

=== MATRIZ DE HOMOLOGACIÓN AL ESQUEMA CANÓNICO ===


,Campo Canónico,Yellow Taxi,Green Taxi,FHVHV
0,airport_fee,Airport_fee,— (no existe),airport_fee
1,congestion_surcharge,congestion_surcharge,congestion_surcharge,congestion_surcharge
2,dropoff_datetime,tpep_dropoff_datetime,lpep_dropoff_datetime,dropoff_datetime
3,dropoff_location_id,DOLocationID,DOLocationID,DOLocationID
4,ehail_fee,— (no existe),ehail_fee,— (no existe)
5,extra_amount,extra,extra,— (no existe)
6,fare_amount,fare_amount,fare_amount,base_passenger_fare
7,improvement_surcharge,improvement_surcharge,improvement_surcharge,— (no existe)
8,mta_tax,mta_tax,mta_tax,sales_tax
9,passenger_count,passenger_count,passenger_count,— (no existe)


In [3]:
# ── 2. Extraer archivos y ejecutar diagnóstico ────────────────────
dataframes_by_service, inventory_records = run_extraction_phase(
    spark, config, PROCESS_ID, logger
)

# Diagnóstico individual por servicio
print('\n=== DIAGNÓSTICO DE ESQUEMAS POR SERVICIO ===')
all_diagnostics = []

for service, file_list in dataframes_by_service.items():
    if not file_list:
        continue
    for file_path, df in file_list:
        diag = diagnose_schema(df, service, '../metadata', logger)
        diag['file'] = file_path.split('/')[-1]
        all_diagnostics.append(diag)

        print(f'\n  Archivo: {diag["file"]}')
        print(f'  Servicio          : {service}')
        print(f'  Columnas reales   : {diag["actual_columns"]}')
        print(f'  Columnas esperadas: {diag["expected_columns"]}')
        print(f'  Ausentes          : {diag["missing_columns"]}')
        print(f'  Extra             : {diag["extra_columns"]}')
        print(f'  Tipo-incompatibles: {diag["type_mismatches"]}')
        print(f'  Recuperable       : {diag["is_recoverable"]}')

2026-06-23 11:05:20 | INFO     | nb02_schema                    | ══════════════════════════════════════════════════════════════════════
2026-06-23 11:05:20 | INFO     | nb02_schema                    |   FASE 1: EXTRACCIÓN  —  process_id: ETL-20260623-110509-331BC2BB
2026-06-23 11:05:20 | INFO     | nb02_schema                    | ══════════════════════════════════════════════════════════════════════
2026-06-23 11:05:20 | INFO     | nb02_schema                    | 
[1.1] Descarga de archivos fuente:
2026-06-23 11:05:20 | INFO     | nb02_schema                    | ── Descargando archivos NYC TLC ───────────────────────────────
2026-06-23 11:05:20 | INFO     | nb02_schema                    |   Servicio: YELLOW
2026-06-23 11:05:20 | INFO     | nb02_schema                    |     [CACHE] yellow_tripdata_2023-01.parquet (45.46 MB) — ya existe, omitiendo descarga
2026-06-23 11:05:20 | INFO     | nb02_schema                    |     [CACHE] yellow_tripdata_2023-02.parquet (45.54 MB) — y

In [4]:
# ── 3. Ejecutar reconstrucción canónica completa (Capa Bronze) ────
bronze_df, diagnostics = run_schema_recovery_phase(
    spark, dataframes_by_service, config, PROCESS_ID, logger
)

if bronze_df:
    total_bronze = bronze_df.count()
    print(f'\n✅ Bronze construido: {total_bronze:,} registros, {len(bronze_df.columns)} columnas')
    print(f'\nColumnas del esquema canónico:')
    for col in bronze_df.columns:
        dtype = dict(bronze_df.dtypes).get(col, '?')
        print(f'  {col:<30} {dtype}')
else:
    print('⚠️ No se pudo construir la capa bronze')

2026-06-23 11:05:31 | INFO     | nb02_schema                    | ══════════════════════════════════════════════════════════════════════
2026-06-23 11:05:31 | INFO     | nb02_schema                    |   FASE 2: DIAGNÓSTICO Y RECONSTRUCCIÓN  —  process_id: ETL-20260623-110509-331BC2BB
2026-06-23 11:05:31 | INFO     | nb02_schema                    | ══════════════════════════════════════════════════════════════════════
2026-06-23 11:05:31 | INFO     | nb02_schema                    | 
[2.1] Procesando servicio: YELLOW (8 archivos)
2026-06-23 11:05:31 | WARNING  | nb02_schema                    |   Diagnóstico yellow: 1 ausentes, 2 extra, 2 tipo-incompatibles
2026-06-23 11:05:32 | INFO     | nb02_schema                    |   [OK]  yellow_tripdata_2020-01.parquet → 6,405,008 registros reconstruidos
2026-06-23 11:05:32 | WARNING  | nb02_schema                    |   Diagnóstico yellow: 1 ausentes, 2 extra, 2 tipo-incompatibles
2026-06-23 11:05:33 | INFO     | nb02_schema                

In [5]:
# ── 4. Distribución de registros por servicio en Bronze ───────────
if bronze_df:
    from pyspark.sql import functions as F
    print('=== DISTRIBUCIÓN EN CAPA BRONZE ===')
    bronze_df.groupBy('service_type') \
             .agg(F.count('*').alias('total_registros'),
                  F.min('pickup_datetime').alias('fecha_min'),
                  F.max('pickup_datetime').alias('fecha_max')) \
             .orderBy('service_type') \
             .show(truncate=False)

    print('\n=== MUESTRA BRONZE (5 filas) ===')
    bronze_df.select('service_type','pickup_datetime','dropoff_datetime',
                     'pickup_location_id','fare_amount','total_amount','source_file') \
             .show(5, truncate=False)

=== DISTRIBUCIÓN EN CAPA BRONZE ===
+------------+---------------+-------------------+-------------------+
|service_type|total_registros|fecha_min          |fecha_max          |
+------------+---------------+-------------------+-------------------+
|fhvhv       |18479031       |2023-01-01 00:00:00|2023-01-31 23:59:59|
|green       |133020         |2008-12-31 23:02:29|2023-03-01 00:01:06|
|yellow      |25890876       |2001-01-01 00:06:49|2023-05-03 23:14:53|
+------------+---------------+-------------------+-------------------+


=== MUESTRA BRONZE (5 filas) ===
+------------+-------------------+-------------------+------------------+-----------+------------+-------------------------------+
|service_type|pickup_datetime    |dropoff_datetime   |pickup_location_id|fare_amount|total_amount|source_file                    |
+------------+-------------------+-------------------+------------------+-----------+------------+-------------------------------+
|yellow      |2020-01-01 00:28:15|2020-

In [6]:
# ── 5. Revisar archivos en cuarentena ────────────────────────────
import glob
quarantine_files = glob.glob('../data/quarantine/files/*.json')
print(f'=== ARCHIVOS EN CUARENTENA: {len(quarantine_files)} ===')

for qf in quarantine_files:
    with open(qf) as f:
        entry = json.load(f)
    print(f'\n  Archivo: {entry.get("file_name")}')
    print(f'  Estado : {entry.get("read_status")}')
    print(f'  Error  : {str(entry.get("error_message", ""))[:150]}')
    print(f'  Acción : {entry.get("recommended_action", "")}')

print(f'\n✅ Fase 2-3 completada — process_id: {PROCESS_ID}')
# ==========================================================
# LIBERACIÓN DE RECURSOS NOTEBOOK 02
# ==========================================================

print("\nLiberando recursos Spark...")

try:
    spark.catalog.clearCache()
    print("✅ Cache Spark liberada")
except Exception as e:
    print(f"⚠ Cache: {e}")

try:
    spark.sparkContext._jvm.java.lang.System.gc()
    print("✅ Garbage Collector JVM ejecutado")
except:
    pass

try:
    spark.stop()
    print("✅ SparkSession detenida")
except Exception as e:
    print(f"⚠ Spark stop: {e}")

print("\n✅ Notebook 02 finalizado correctamente")

=== ARCHIVOS EN CUARENTENA: 18 ===

  Archivo: ARROW-GH-41317.parquet
  Estado : RECOVERABLE_SCHEMA_MISMATCH
  Error  : An error occurred while calling o73.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, 
  Acción : Aplicar reconstrucciÃ³n de esquema canÃ³nico en Fase 2.

  Archivo: ARROW-GH-41317.parquet
  Estado : RECOVERABLE_SCHEMA_MISMATCH
  Error  : An error occurred while calling o73.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, 
  Acción : Aplicar reconstrucciÃ³n de esquema canÃ³nico en Fase 2.

  Archivo: ARROW-GH-41317.parquet
  Estado : RECOVERABLE_SCHEMA_MISMATCH
  Error  : An error occurred while calling o73.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, 
  Acción : Aplicar reconstrucciÃ³n de esquema canÃ³nico en Fase 2.

  Archivo: ARROW-GH-41317.parquet
  Estado : RECOVERABL